### RAG Pipeline


#### Data Ingestion to Vector DB


In [21]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from pathlib import Path

In [22]:
# Read all pdf files from a directory and load them as documents
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    return all_documents

all_pdfs = process_all_pdfs("../data/pdf")    



Found 6 PDF files to process

Processing: cyber.pdf
Loaded 29 pages

Processing: hardware.pdf
Loaded 1 pages

Processing: io-t.pdf
Loaded 7 pages

Processing: iot.pdf
Loaded 1 pages

Processing: javascript.pdf
Loaded 1 pages

Processing: nlp.pdf
Loaded 1 pages


In [23]:
all_pdfs

[Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows); modified using iText® 5.3.5 ©2000-2012 1T3XT BVBA (SPRINGER SBM; licensed version)', 'creator': 'Springer', 'creationdate': '2020-06-29T13:18:16+05:30', 'keywords': 'Cybersecurity,Machine learning,Data science,Decision making,Cyber-attack,Security modeling,Intrusion detection,Cyber threat intelligence', 'crossmarkdomains[1]': 'springer.com', 'moddate': '2020-07-01T08:42:45+02:00', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Journal of Big Data, https://doi.org/10.1186/s40537-020-00318-5', 'author': 'Iqbal H. Sarker', 'title': 'Cybersecurity data science: an overview from machine learning perspective', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'doi': '10.1186/s40537-020-00318-5', 'crossmarkdomains[2]': 'springerlink.com', 'source': '..\\data\\pdf\\cyber.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1', 'source_file': 'cyber.pdf', 'file_type': 'pdf'}, page_content='Cybersecurity data scien

In [24]:
### Text Splitter into Chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #show example chunk
    if split_docs:
        print("\nExample chunk:")
        print(split_docs[0].page_content[:200])  # Print first 200 characters of the first chunk
        print("\nMetadata:", split_docs[0].metadata)    

    return split_docs

In [25]:
chunks = split_documents(all_pdfs)
chunks

Split 40 documents into 195 chunks

Example chunk:
Cybersecurity data science: an overview 
from machine learning perspective
Iqbal H. Sarker1,2* , A. S. M. Kayes3, Shahriar Badsha4, Hamed Alqahtani5, Paul Watters3 and Alex Ng3
Introduction
Due to the

Metadata: {'producer': 'Acrobat Distiller 10.1.8 (Windows); modified using iText® 5.3.5 ©2000-2012 1T3XT BVBA (SPRINGER SBM; licensed version)', 'creator': 'Springer', 'creationdate': '2020-06-29T13:18:16+05:30', 'keywords': 'Cybersecurity,Machine learning,Data science,Decision making,Cyber-attack,Security modeling,Intrusion detection,Cyber threat intelligence', 'crossmarkdomains[1]': 'springer.com', 'moddate': '2020-07-01T08:42:45+02:00', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Journal of Big Data, https://doi.org/10.1186/s40537-020-00318-5', 'author': 'Iqbal H. Sarker', 'title': 'Cybersecurity data science: an overview from machine learning perspective', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'doi': '10

[Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows); modified using iText® 5.3.5 ©2000-2012 1T3XT BVBA (SPRINGER SBM; licensed version)', 'creator': 'Springer', 'creationdate': '2020-06-29T13:18:16+05:30', 'keywords': 'Cybersecurity,Machine learning,Data science,Decision making,Cyber-attack,Security modeling,Intrusion detection,Cyber threat intelligence', 'crossmarkdomains[1]': 'springer.com', 'moddate': '2020-07-01T08:42:45+02:00', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Journal of Big Data, https://doi.org/10.1186/s40537-020-00318-5', 'author': 'Iqbal H. Sarker', 'title': 'Cybersecurity data science: an overview from machine learning perspective', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'doi': '10.1186/s40537-020-00318-5', 'crossmarkdomains[2]': 'springerlink.com', 'source': '..\\data\\pdf\\cyber.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1', 'source_file': 'cyber.pdf', 'file_type': 'pdf'}, page_content='Cybersecurity data scien

In [26]:
import numpy as np  
from sentence_transformers import SentenceTransformer
import faiss
import chromadb
import uuid
from typing import List, Any, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [27]:
class EmbeddingManager:
    """handle document embeddingg generation using sentence-transformers"""
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully : Embedding dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")       
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")

        print(f"generate embedding for {len(texts)} texts...") 
        embedding = self.model.encode(texts, show_progress_bar = True)
        print("generate embedding with shape: {embeddings.shape}")
        return embedding

#initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001E240B0B050>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: f12dcc99-ff1a-4a05-b9e7-b7b005f845c4)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Model loaded successfully : Embedding dimension: 384


#### VectorStore

In [28]:
class VectorStore:
    """manage document embedding in ChromaDB vector store"""
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(name=self.collection_name, metadata={"description": "PDF document for RAG"})

            print(f"Vector store initialized at {self.persist_directory} with collection '{self.collection_name}'")
            print(f"Current number of documents in store: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise e

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        print(f"Adding {len(documents)} documents to vector store...")
        #prepare document for chrmadb
        ids = []
        metadatas = []
        document_texts = [] 
        embedding_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #generate unique id for each document
            doc_id =  f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            #prepare metadata and text
            metadata = dict(doc.metadata)  
            metadata['doc_index'] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            #document content
            document_texts.append(doc.page_content)

            #embedding
            embedding_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                documents=document_texts,
                metadatas=metadatas,
                embeddings=embedding_list
            )
            print(f"Successfully added {len(documents)} documents. Total documents in store: {self.collection.count()}")    
        except Exception as e:  
            print(f"Error adding documents to vector store: {e}")
            raise e

vector_store = VectorStore()
vector_store



Vector store initialized at ../data/vector_store with collection 'pdf_documents'
Current number of documents in store: 195


In [29]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 10.1.8 (Windows); modified using iText® 5.3.5 ©2000-2012 1T3XT BVBA (SPRINGER SBM; licensed version)', 'creator': 'Springer', 'creationdate': '2020-06-29T13:18:16+05:30', 'keywords': 'Cybersecurity,Machine learning,Data science,Decision making,Cyber-attack,Security modeling,Intrusion detection,Cyber threat intelligence', 'crossmarkdomains[1]': 'springer.com', 'moddate': '2020-07-01T08:42:45+02:00', 'crossmarkmajorversiondate': '2010-04-23', 'subject': 'Journal of Big Data, https://doi.org/10.1186/s40537-020-00318-5', 'author': 'Iqbal H. Sarker', 'title': 'Cybersecurity data science: an overview from machine learning perspective', 'crossmarkdomainexclusive': 'true', 'robots': 'noindex', 'doi': '10.1186/s40537-020-00318-5', 'crossmarkdomains[2]': 'springerlink.com', 'source': '..\\data\\pdf\\cyber.pdf', 'total_pages': 29, 'page': 0, 'page_label': '1', 'source_file': 'cyber.pdf', 'file_type': 'pdf'}, page_content='Cybersecurity data scien

In [30]:
### Convert text to embeddings
texts = [doc.page_content for doc in chunks]
# texts
embeddings = embedding_manager.generate_embeddings(texts)

#store in vector db
vector_store.add_documents(chunks, embeddings)


generate embedding for 195 texts...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches: 100%|██████████| 7/7 [01:24<00:00, 12.01s/it]


generate embedding with shape: {embeddings.shape}
Adding 195 documents to vector store...
Successfully added 195 documents. Total documents in store: 390


#### Retrieval Pipeline from VectorDB

In [31]:
class RAGRetrieval:
    """Retrieval Augmented Generation Retrieval Pipeline"""
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve top-k relevant documents for the given query."""
        print(f"Generating embedding for query: {query}")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        # Search In Vector db
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]


                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score
                    similarity_score = 1 - distance  
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })


                        print(f"Retrieved {len(retrieved_docs)} documents after applying filtering.")
                    else:
                        print("No document find")

                    return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_restriever = RAGRetrieval(vector_store, embedding_manager)
rag_restriever                        



In [32]:
rag_restriever.retrieve("How put Machine learning in cyber")

Generating embedding for query: How put Machine learning in cyber
Top K: 5, Score Threshold: 0.0
generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.61it/s]

generate embedding with shape: {embeddings.shape}
Retrieved 1 documents after applying filtering.


[{'id': 'doc_54b426ca_52',
  'content': 'Machine learning (ML) is typically considered as a branch of “ Artificial Intelligence” , \nwhich is closely related to computational statistics, data mining and analytics, data sci\n-\nence, particularly focusing on making the computers to learn from data [82, 83]. Thus, \nmachine learning models typically comprise of a set of rules, methods, or complex \n“transfer functions” that can be applied to find interesting data patterns, or to recognize \nor predict behavior [84], which could play an important role in the area of cybersecurity. \nIn the following, we discuss different methods that can be used to solve machine learn\n-\ning tasks and how they are related to cybersecurity tasks.\nSupervised learning\nSupervised learning is performed when specific targets are defined to reach from a cer -\ntain set of inputs, i.e., task-driven approach. In the area of machine learning, the most \npopular supervised learning techniques are known as classif

In [33]:
rag_restriever.retrieve("Micro-Electro-Mechanical Systems (MEMS)Technologies")

Generating embedding for query: Micro-Electro-Mechanical Systems (MEMS)Technologies
Top K: 5, Score Threshold: 0.0
generate embedding for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.11it/s]

generate embedding with shape: {embeddings.shape}
Retrieved 1 documents after applying filtering.


[{'id': 'doc_aed8d9cc_168',
  'content': 'This technology realizes smaller and improved version of the things\nthat are interconnected. It can decrease the consumption of a sys-\ntem by enabling the development of devices in nano meters scale\nwhich can be used as a sensor and an actuator just like a normal\ndevice. Such a nano device is made from nano components and\nthe resulting network deﬁnes a new networking paradigm which is\nInternet of Nano-Things [38].\n4.6 Micro-Electro-Mechanical Systems (MEMS)\nTechnologies\nMEMS are a combination of electric and mechanical components\nworking together to provide several applications including sensing\nand actuating which are already being commercially used in many\nﬁeld in the form of transducers and accelerometers etc. MEMS\ncombined with Nano technologies are a cost-effective solution for\nimprovising the communication system of IoT and other advan-\ntages like size reduction of sensors and actuators, integrated ubiq-\nuitous computing d

#### LLM Pipeline 

In [1]:
print("RAG Retrieval successful.")

RAG Retrieval successful.


In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=1,
    max_output_tokens=99,
    timeout=15,
    max_retries=1,
)

In [91]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I hate programming lanuae JAVA."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--6444970d-55d7-4907-8b13-44b72dadf8c8-0', usage_metadata={'input_tokens': 25, 'output_tokens': 398, 'total_tokens': 423, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 398}})

In [1]:
# simple RAG function
def rag_simple(query,rag_restriever,llm,top_k=5):
    #retrieve the context
    results = rag_restriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else "No relevant found."
    if not context:
        return "No relevant context found."

    #prepare the prompt
    prompt = f"""You are an AI assistant. Use the following context to answer the question.
        Context:
        {context}

        Question: {query}
        
        Answer:"""    


    response = llm.invoke(prompt.format(context=context, query=query))   
    return response.content 

In [2]:
answer = rag_simple("Explain Micro-Electro-Mechanical Systems (MEMS) Technologies", rag_restriever, llm)
answer
print(answer)

NameError: name 'rag_restriever' is not defined